In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from scipy.stats import norm
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, backend as K
from keras.saving import register_keras_serializable

K.clear_session()

In [ ]:
# Define the reparameterization trick function

def reparameterization_trick(mean, log_var):
  epsilon = tf.random.normal(shape=tf.shape(mean))
  return mean + tf.exp(0.5 * log_var) * epsilon



def build_encoder(input_shape, latent_dim):

  """

  Builds an encoder model with Conv2D, GRU, mean, and log variance outputs.
  Parameters:
  input_shape (tuple): Shape of the input image, e.g., (36, 36, 1).
  latent_dim (int): Dimension of the latent representation.
  Returns:
  encoder (tf.keras.Model): Encoder model.

  """

  encoder_inputs = layers.Input(shape=input_shape)

  # Conv2D layers with batch normalization
  x = layers.Conv2D(32, kernel_size=(3, 3), activation='leaky_relu', padding='same')(encoder_inputs)
  x = layers.BatchNormalization()(x)
  x = layers.Dropout(0.2)(x)
  x = layers.Conv2D(64, kernel_size=(3, 3), activation='leaky_relu', padding='same')(x)
  x = layers.BatchNormalization()(x)
  x = layers.Dropout(0.2)(x)
  x = layers.Conv2D(128, kernel_size=(3, 3), activation='leaky_relu', padding='same')(x)
  x = layers.BatchNormalization()(x)
  x = layers.Conv2D(256, kernel_size=(3, 3), activation='leaky_relu', padding='same')(x)
  x = layers.BatchNormalization()(x)

  # Flatten the output
  x = layers.Flatten()(x)

  # GRU layer to capture sequential patterns
  x = layers.Reshape((-1, x.shape[-1]))(x)
  x = layers.GRU(128)(x)

  # Output mean and log variance
  mean = layers.Dense(latent_dim)(x)
  log_var = layers.Dense(latent_dim)(x)

  # Define encoder model
  encoder = models.Model(encoder_inputs, [mean, log_var], name="encoder")
  return encoder



def build_decoder(latent_dim, output_shape):

  """
  Builds a decoder model.
  Parameters:
  latent_dim (int): Dimension of the latent representation.
  output_shape (tuple): Shape of the output image, e.g., (36, 36, 1).
  Returns:
  decoder (tf.keras.Model): Decoder model.

  """

  decoder_inputs = layers.Input(shape=(latent_dim,))

  # Dense layer to expand latent space to a suitable shape for decoding
  x = layers.Dense(9 * 9 * 128, activation='relu')(decoder_inputs)
  x = layers.Reshape((9, 9, 128))(x)


  # Conv2DTranspose layers with batch normalization to upsample and reconstruct the image
  x = layers.Conv2DTranspose(128, kernel_size=(3, 3), strides=(2, 2), padding='same', activation='leaky_relu')(x)
  x = layers.BatchNormalization()(x)
  x = layers.Conv2DTranspose(64, kernel_size=(3, 3), strides=(2, 2), padding='same', activation='leaky_relu')(x)
  x = layers.BatchNormalization()(x)
  x = layers.Dropout(0.2)(x)
  x = layers.Conv2DTranspose(32, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='leaky_relu')(x)
  x = layers.BatchNormalization()(x)
  x = layers.Dropout(0.2)(x)



  # Final Conv2DTranspose layer to match output shape and produce the final image
  decoder_outputs = layers.Conv2DTranspose(output_shape[-1], kernel_size=(3, 3), padding='same', activation='sigmoid')(x)



  # Define decoder model
  decoder = models.Model(decoder_inputs, decoder_outputs, name="decoder")
  return decoder


# Specify input shape and latent dimension
input_shape = (36, 36, 1)
latent_dim = 10



# Build encoder and decoder models
encoder = build_encoder(input_shape, latent_dim)
decoder = build_decoder(latent_dim, input_shape)



# Define a VAE model using the encoder and decoder
class VAE(tf.keras.Model):

  def __init__(self, encoder, decoder, **kwargs):
    super(VAE, self).__init__(**kwargs)
    self.encoder = encoder
    self.decoder = decoder



  def call(self, inputs):

    # Encode inputs to mean and log variance
    mean, log_var = self.encoder(inputs)

    # Use reparameterization trick to sample from the Gaussian distribution
    z = reparameterization_trick(mean, log_var)

    # Decode the sampled latent vector
    reconstructed = self.decoder(z)
    return reconstructed, mean, log_var



# Instantiate the VAE model
vae = VAE(encoder, decoder)